In [0]:
%sql
SELECT * FROM ECOMMERCE.BRONZE.clickstream LIMIT 10;

## Load Libraries for future usage 

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col


In [0]:
from pyspark.sql.functions import col, when

clickstream_df = (
    spark.read.table("ECOMMERCE.BRONZE.CLICKSTREAM")
    .withColumn("event_time", col("event_time").cast("timestamp"))
    .dropDuplicates()
)

clickstream_df = clickstream_df.withColumn(
    "customer_id", when(col("customer_id").isNull(), -1.0).otherwise(col("customer_id"))
)

clickstream_df = clickstream_df.withColumn(
    "customer_id",
    when(
        col("customer_id").rlike(r"^[0-9]*\.?[0-9]+$"),
        col("customer_id").cast("double")
    ).otherwise(-1).cast("int")
)
display(clickstream_df)

## Validate timestamps

In [0]:
clickstream_df = clickstream_df.withColumn(
    "valid_event_time", when(col("event_time").isNotNull(), 1).otherwise(0)
)

In [0]:
clickstream_df = clickstream_df.withColumn("event_type", upper(trim(col("event_type"))))

In [0]:
clickstream_df = clickstream_df.withColumn(
    "valid_product_id",
    when(
        col("product_id").isNull() | col("product_id").cast("int").isNotNull(), 1
    ).otherwise(0),
)

In [0]:
clickstream_df = clickstream_df.withColumn(
    "valid_customer_id", when(col("customer_id") != -1, 1).otherwise(0)
)

In [0]:
clickstream_df = clickstream_df.withColumn(
    "event_date",
    to_date(col("event_time"))
)

In [0]:
clickstream_df = clickstream_df.withColumn(
    "event_hour",
    hour(col("event_time"))
)


In [0]:
clickstream_df = clickstream_df.withColumn(
    "is_anonymous_session",
    when(col("customer_id")==-1, 1).otherwise(0)
)

In [0]:
display(clickstream_df)

In [0]:
clickstream_df.write.format("delta").mode("append").saveAsTable(
    "ecommerce.silver.clickstream"
)

In [0]:
%sql
SELECT * FROM ECOMMERCE.SILVER.CLICKSTREAM LIMIT 100;